# Module 5: Machine Learning Forecasting (Global Models)

This notebook trains **global ML models** across all SKUs.

## What you'll do
- Load the feature-ready dataset (or regenerate it from raw data)
- Create a horizon target (predict demand H days ahead)
- Train global models (Ridge, RandomForest; optional LightGBM/XGBoost)
- Evaluate with MAE/RMSE/WAPE/sMAPE
- Inspect feature importance (permutation importance)
- Compare vs baselines conceptually (Module 3)


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

from src.data.loaders import load_sales_data
from src.data.cleaners import fill_missing_dates, handle_outliers
from src.features.engineering import (
    create_temporal_features,
    create_rolling_features,
    create_promotion_features,
    create_price_features,
    create_interaction_features,
)
from src.features.lag_features import create_lag_features

from src.models.ml_models import (
    make_horizon_target,
    time_cutoff_split,
    build_supervised_dataset,
    make_model,
    build_sklearn_pipeline,
    permutation_importance_topk,
)
from src.evaluation.metrics import mae, rmse, wape, smape

print('Imports OK')


Imports OK


## Load feature-ready data (or regenerate)

Preferred:
- `data/processed/featured_sales_data.csv`

If missing, we regenerate from:
- `data/raw/sample_sales.csv`

This keeps Module 5 runnable even if you didn’t persist Module 2 outputs.


In [2]:
processed_candidates = [
    Path('../data/processed/featured_sales_data.csv'),
    Path('data/processed/featured_sales_data.csv'),
]

processed_path = next((p for p in processed_candidates if p.exists()), None)

if processed_path is not None:
    df = pd.read_csv(processed_path, parse_dates=['date'])
    print('Loaded processed:', processed_path)
else:
    raw_candidates = [
        Path('../data/raw/sample_sales.csv'),
        Path('data/raw/sample_sales.csv'),
    ]
    raw_path = next((p for p in raw_candidates if p.exists()), None)
    if raw_path is None:
        raise FileNotFoundError('Could not find raw sample data at data/raw/sample_sales.csv')

    raw = load_sales_data(raw_path)
    print('Loaded raw:', raw_path, raw.shape)

    # Minimal re-run of Module 2 pipeline
    df = fill_missing_dates(raw, date_col='date', group_col='sku_id', fill_value=0, fill_col='units_sold')
    df = handle_outliers(df, column='units_sold', method='cap', lower_percentile=0.01, upper_percentile=0.99)

    df = create_temporal_features(df, date_col='date')
    df = create_lag_features(df, value_col='units_sold', group_col='sku_id', date_col='date', lags=[1, 7, 14, 30, 90])
    df = create_rolling_features(df, value_col='units_sold', group_col='sku_id', date_col='date', windows=[7, 14, 30, 90])
    df = create_promotion_features(df, promo_col='promotion_flag', group_col='sku_id', date_col='date')
    df = create_price_features(df, price_col='price', group_col='sku_id', date_col='date')
    df = create_interaction_features(df)

    print('Regenerated features:', df.shape)

df.head()


Loaded raw: ..\data\raw\sample_sales.csv (365000, 9)
Regenerated features: (365000, 65)


,sku_id,date,category,subcategory,price,units_sold,revenue,promotion_flag,stock_available,year,...,price_rolling_mean_7,price_relative_to_avg_7,price_rolling_mean_30,price_relative_to_avg_30,price_rolling_std_7,month_day_interaction,holiday_weekend,promo_weekend,is_peak_period,quarter_day_interaction
0,SKU001,2023-12-18,Electronics,Accessories,155.30,19,2950.62,0,8,2023,...,155.300000,1.000000,155.300000,1.000000,NaN,0,0,0,0,0
1,SKU001,2023-12-19,Electronics,Accessories,158.31,26,4116.01,0,103,2023,...,156.805000,1.009598,156.805000,1.009598,2.128391,12,0,0,0,4
2,SKU001,2023-12-20,Electronics,Accessories,127.46,35,4461.21,1,162,2023,...,147.023333,0.866937,147.023333,0.866937,17.009057,24,0,0,0,8
3,SKU001,2023-12-21,Electronics,Accessories,154.27,15,2314.03,0,82,2023,...,148.835000,1.036517,148.835000,1.036517,14.352720,36,0,0,0,12
4,SKU001,2023-12-22,Electronics,Accessories,163.09,26,4240.44,0,165,2023,...,151.686000,1.075182,151.686000,1.075182,13.969303,48,0,0,0,16


## Create a horizon target (supervised learning)

We predict **units_sold at t+H** using features at time t.

Important: we will **exclude** contemporaneous `units_sold` from the feature set and rely on lag/rolling features instead.


In [3]:
HORIZON = 14

# Add future label per SKU
labeled = make_horizon_target(df, group_col='sku_id', date_col='date', target_col='units_sold', horizon=HORIZON, target_name='y')

# Drop rows where target is not available (tail of each SKU)
print('Before dropna:', labeled.shape)
labeled = labeled.dropna(subset=['y']).reset_index(drop=True)
print('After dropna:', labeled.shape)

labeled[['sku_id','date','units_sold','y']].head(10)


Before dropna: (365000, 66)
After dropna: (358000, 66)


,sku_id,date,units_sold,y
0,SKU001,2023-12-18,19,12.0
1,SKU001,2023-12-19,26,13.0
2,SKU001,2023-12-20,35,7.0
3,SKU001,2023-12-21,15,13.0
4,SKU001,2023-12-22,26,16.0
5,SKU001,2023-12-23,0,33.0
6,SKU001,2023-12-24,0,0.0
7,SKU001,2023-12-25,24,13.0
8,SKU001,2023-12-26,20,6.0
9,SKU001,2023-12-27,22,17.0


## Time-based validation split

We’ll validate on the last `HORIZON` days (simple holdout).


In [4]:
max_date = labeled['date'].max()
cutoff = max_date - pd.Timedelta(days=HORIZON)

train_df, valid_df = time_cutoff_split(labeled, date_col='date', cutoff=cutoff)

# Keep only the validation window we care about (next HORIZON days)
valid_df = valid_df[(valid_df['date'] > cutoff) & (valid_df['date'] <= cutoff + pd.Timedelta(days=HORIZON))].copy()

print('Cutoff:', cutoff.date())
print('Train:', train_df.shape, 'Valid:', valid_df.shape)
print('Valid date range:', valid_df['date'].min().date(), '->', valid_df['date'].max().date())


Cutoff: 2025-11-18
Train: (351000, 66) Valid: (7000, 66)
Valid date range: 2025-11-19 -> 2025-12-02


## Build X/y and train models

We’ll drop columns that are unsafe/leaky for a strict “features at time t” setup:
- `units_sold` (use lags/rolling instead)
- `revenue` (often derived from same-day demand; optional)

We keep identifiers like `sku_id`, `category`, `subcategory` and encode them.


In [5]:
DROP_COLS = ['units_sold', 'revenue']
CAT_COLS = [c for c in ['sku_id', 'category', 'subcategory'] if c in train_df.columns]

train_ds = build_supervised_dataset(train_df, date_col='date', target_name='y', drop_cols=DROP_COLS, categorical_cols=CAT_COLS)
valid_ds = build_supervised_dataset(valid_df, date_col='date', target_name='y', drop_cols=DROP_COLS, categorical_cols=CAT_COLS)

print('Categorical:', train_ds.categorical_cols)
print('Numeric sample:', train_ds.numeric_cols[:10], '...')
print('X_train:', train_ds.X.shape, 'X_valid:', valid_ds.X.shape)


Categorical: ['sku_id', 'category', 'subcategory']
Numeric sample: ['price', 'promotion_flag', 'stock_available', 'year', 'month', 'day', 'day_of_week', 'day_of_month', 'week', 'quarter'] ...
X_train: (351000, 62) X_valid: (7000, 62)


In [6]:
def eval_predictions(y_true, y_pred) -> dict:
    return {
        'MAE': mae(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'WAPE': wape(y_true, y_pred),
        'sMAPE': smape(y_true, y_pred),
    }

models_to_try = ['ridge', 'rf', 'lgbm', 'xgb']
results = []
fitted = {}

for kind in models_to_try:
    try:
        model = make_model(kind)
        pipe = build_sklearn_pipeline(model, train_ds.categorical_cols, train_ds.numeric_cols)
        pipe.fit(train_ds.X, train_ds.y)

        pred = pipe.predict(valid_ds.X)
        m = eval_predictions(valid_ds.y, pred)
        m['model'] = kind
        results.append(m)
        fitted[kind] = pipe
        print(f"✓ Trained {kind}: WAPE={m['WAPE']:.4f}")
    except Exception as e:
        print(f"Skipping {kind}: {e}")

summary = pd.DataFrame(results).set_index('model').sort_values('WAPE')
summary


Skipping ridge: Input X contains NaN.
Ridge does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values
Skipping rf: Input X contains NaN.
RandomForestRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using

,MAE,RMSE,WAPE,sMAPE
model,,,,
lgbm,18.828100,26.585742,0.407975,0.529131
xgb,18.861018,26.704547,0.408688,0.529133


## Feature importance (permutation)

We compute permutation importance on the validation set for the best available model.

Note: this can be slow on large datasets; we’ll sample the validation set if needed.


In [7]:
if len(summary) == 0:
    raise RuntimeError('No ML models trained successfully (install sklearn/lightgbm/xgboost as needed).')

best_kind = summary.index[0]
best_pipe = fitted[best_kind]
print('Best model:', best_kind)

# Sample validation rows for speed
n = min(5000, len(valid_ds.X))
sample_idx = np.random.RandomState(42).choice(len(valid_ds.X), size=n, replace=False)
Xv = valid_ds.X.iloc[sample_idx]
yv = valid_ds.y.iloc[sample_idx]


Best model: lgbm


In [8]:
imp = permutation_importance_topk(best_pipe, Xv, yv, k=25)
imp

,feature,importance
0,f45,2.486268
1,f42,2.096700
2,f40,1.610117
3,f41,0.626460
4,f37,0.398855
5,f36,0.288006
6,f9,0.201957
7,f27,0.136113
8,f32,0.126667
9,f35,0.125398


In [9]:
# Save results
out_dir = Path('../outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)

summary_out = out_dir / 'module5_ml_summary.csv'
summary.reset_index().to_csv(summary_out, index=False)
print('Saved:', summary_out)

imp_out = out_dir / 'module5_ml_feature_importance.csv'
imp.to_csv(imp_out, index=False)
print('Saved:', imp_out)


Saved: ..\outputs\reports\module5_ml_summary.csv
Saved: ..\outputs\reports\module5_ml_feature_importance.csv
